<a href="https://colab.research.google.com/github/Pratyakshk05/deep-learning/blob/main/lab9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torch torchvision wandb matplotlib

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader
import wandb
import os

# ---------------- CONFIG ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 128
z_dim = 100
epochs = 20
lr = 0.0002

USE_DCGAN = True
LOSS_TYPE = "BCE"      # BCE / LSGAN / WGAN
OPTIMIZER = "Adam"     # SGD / RMSprop / Adam

os.makedirs("outputs", exist_ok=True)

wandb.init(project="GAN-Full-Experiment")

wandb.config.update({
    "architecture": "DCGAN" if USE_DCGAN else "Vanilla",
    "loss": LOSS_TYPE,
    "optimizer": OPTIMIZER,
    "epochs": epochs
})

# ---------------- DATA ----------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# ---------------- MODELS ----------------

# Vanilla GAN
class VanillaGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x).view(-1, 1, 28, 28)


class VanillaDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.net(x)


# DCGAN
class DCGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 128, 7, 1, 0),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 1, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)


class DCDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(128*7*7, 1)
        )

    def forward(self, x):
        return self.net(x)


# ---------------- SELECT MODEL ----------------
if USE_DCGAN:
    G = DCGenerator().to(device)
    D = DCDiscriminator().to(device)
else:
    G = VanillaGenerator().to(device)
    D = VanillaDiscriminator().to(device)

# ---------------- LOSS ----------------
if LOSS_TYPE == "BCE":
    criterion = nn.BCEWithLogitsLoss()
elif LOSS_TYPE == "LSGAN":
    criterion = nn.MSELoss()

# ---------------- OPTIMIZER ----------------
def get_optimizer(model):
    if OPTIMIZER == "SGD":
        return optim.SGD(model.parameters(), lr=lr)
    elif OPTIMIZER == "RMSprop":
        return optim.RMSprop(model.parameters(), lr=lr)
    else:
        return optim.Adam(model.parameters(), lr=lr, betas=(0.5, 0.999))

opt_G = get_optimizer(G)
opt_D = get_optimizer(D)

fixed_noise = torch.randn(64, z_dim, 1, 1).to(device)

# ---------------- TRAIN ----------------
for epoch in range(epochs):
    for real, _ in loader:
        real = real.to(device)
        batch = real.size(0)

        # Noise
        if USE_DCGAN:
            noise = torch.randn(batch, z_dim, 1, 1).to(device)
        else:
            noise = torch.randn(batch, z_dim).to(device)

        fake = G(noise)

        # -------- Train Discriminator --------
        if LOSS_TYPE == "WGAN":
            loss_D = -torch.mean(D(real)) + torch.mean(D(fake.detach()))

            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

            # Weight Clipping (IMPORTANT)
            for p in D.parameters():
                p.data.clamp_(-0.01, 0.01)

        else:
            real_labels = torch.ones(batch, 1).to(device)
            fake_labels = torch.zeros(batch, 1).to(device)

            D_real = D(real)
            D_fake = D(fake.detach())

            loss_D = criterion(D_real, real_labels) + criterion(D_fake, fake_labels)

            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

        # -------- Train Generator --------
        if LOSS_TYPE == "WGAN":
            loss_G = -torch.mean(D(fake))
        else:
            output = D(fake)
            loss_G = criterion(output, real_labels)

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

    # -------- LOGGING --------
    wandb.log({
        "Generator Loss": loss_G.item(),
        "Discriminator Loss": loss_D.item()
    })

    print(f"Epoch {epoch+1}/{epochs} | G: {loss_G.item():.4f} | D: {loss_D.item():.4f}")

    # -------- SAVE IMAGES --------
    with torch.no_grad():
        fake = G(fixed_noise).detach().cpu()
        utils.save_image(fake, f"outputs/epoch_{epoch+1}.png", normalize=True)
        wandb.log({"Generated Images": [wandb.Image(f"outputs/epoch_{epoch+1}.png")]})

# ---------------- SAVE MODELS ----------------
torch.save(G.state_dict(), "generator.pth")
torch.save(D.state_dict(), "discriminator.pth")

print("✅ Training Complete")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pratyakshk05 (pratyakshk05-delhi-technological-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 26.4M/26.4M [00:00<00:00, 115MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 4.16MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 60.8MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.4MB/s]


Epoch 1/20 | G: 2.2380 | D: 0.5171
Epoch 2/20 | G: 0.8462 | D: 1.1521
Epoch 3/20 | G: 1.8034 | D: 1.0941
Epoch 4/20 | G: 1.1558 | D: 0.8788
Epoch 5/20 | G: 0.9037 | D: 1.1714
Epoch 6/20 | G: 1.6171 | D: 0.9017
Epoch 7/20 | G: 1.0967 | D: 1.0206
Epoch 8/20 | G: 0.9126 | D: 0.8796
Epoch 9/20 | G: 1.0140 | D: 0.8945
Epoch 10/20 | G: 1.5117 | D: 1.0358
Epoch 11/20 | G: 1.5319 | D: 0.8831
Epoch 12/20 | G: 0.7480 | D: 1.0851
Epoch 13/20 | G: 1.5157 | D: 1.0261
Epoch 14/20 | G: 1.0964 | D: 0.9160
Epoch 15/20 | G: 1.7058 | D: 1.0036
Epoch 16/20 | G: 1.0744 | D: 0.9663
Epoch 17/20 | G: 1.5966 | D: 0.8349
Epoch 18/20 | G: 1.1943 | D: 0.9129
Epoch 19/20 | G: 1.1710 | D: 0.9030
Epoch 20/20 | G: 1.2714 | D: 0.8765
✅ Training Complete
